In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# 1º CÓDIGO: Geração do Banco de Dados CSV (Google Colab)

import numpy as np
import pandas as pd

# 1. Configurar semente aleatória para reprodutibilidade
np.random.seed(42)

# 2. Simular dados de profundidade das rugas após 30 dias (em milímetros)
dados_creme = np.random.normal(loc=1.85, scale=0.25, size=30)
dados_placebo = np.random.normal(loc=2.15, scale=0.30, size=30)

# 3. Criar DataFrame estruturado
df_estudo = pd.DataFrame({
    'ID_Voluntaria': range(1, 61),
    'Grupo': ['Creme Antirrugas'] * 30 + ['Placebo'] * 30,
    'Profundidade_Rugas_mm': np.concatenate([dados_creme, dados_placebo])
})

# 4. Exportar para arquivo CSV
df_estudo.to_csv('dados_estudo_cosmeticos.csv', index=False)
print("Arquivo 'dados_estudo_cosmeticos.csv' gerado com sucesso!")

Arquivo 'dados_estudo_cosmeticos.csv' gerado com sucesso!


In [3]:
# 2º CÓDIGO: Análise Estatística

import os
import gdown
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg') # Configuração para ambiente sem interface gráfica integrada
import matplotlib.pyplot as plt
import seaborn as sns

# Para este script, verificamos se o CSV já foi baixado/gerado localmente. Caso contrário, os dados são simulados.
if os.path.exists('dados_estudo_cosmeticos.csv'):
    df_experimento = pd.read_csv('dados_estudo_cosmeticos.csv')
else:
    np.random.seed(42)
    dados_creme = np.random.normal(loc=1.85, scale=0.25, size=30)
    dados_placebo = np.random.normal(loc=2.15, scale=0.30, size=30)
    df_experimento = pd.DataFrame({
        'Grupo': ['Creme Antirrugas'] * 30 + ['Placebo'] * 30,
        'Profundidade_Rugas_mm': np.concatenate([dados_creme, dados_placebo])
    })

# Separação dos grupos em arrays para as funções estatísticas
grupo_creme = df_experimento[df_experimento['Grupo'] == 'Creme Antirrugas']['Profundidade_Rugas_mm'].values
grupo_placebo = df_experimento[df_experimento['Grupo'] == 'Placebo']['Profundidade_Rugas_mm'].values

# 2. Análise Descritiva dos Dados
print("=== Estatísticas Descritivas ===")
print(df_experimento.groupby('Grupo')['Profundidade_Rugas_mm'].describe())
print("\n")

# 3. Verificação de Pressupostos Estatísticos
print("=== Verificação de Pressupostos ===")
# A) Teste de Normalidade de Shapiro-Wilk (Ideal para amostras pequenas/médias)
# H0: Os dados seguem uma distribuição normal
stat_sh_creme, p_sh_creme = stats.shapiro(grupo_creme)
stat_sh_placebo, p_sh_placebo = stats.shapiro(grupo_placebo)
print(f"Normalidade - Grupo Creme (p-value): {p_sh_creme:.4f}")
print(f"Normalidade - Grupo Placebo (p-value): {p_sh_placebo:.4f}")

# B) Teste de Igualdade de Variâncias (Levene)
# H0: As variâncias dos dois grupos são homogêneas
stat_lev, p_lev = stats.levene(grupo_creme, grupo_placebo)
print(f"Igualdade de Variâncias (p-value): {p_lev:.4f}")
print("\n")

# 4. Execução do Teste t de Student para Amostras Independentes
# Hipótese Nula (H0): media_creme = media_placebo (O creme não tem efeito na redução de rugas)
# Hipótese Alternativa (H1): media_creme < media_placebo (O creme é eficaz e reduz a profundidade das rugas)

# Verifica se o p-value do teste de Levene valida a premissa de variâncias iguais (alfa = 0.05)
variancias_iguais = bool(p_lev >= 0.05) 

t_stat, p_value_unilateral = stats.ttest_ind(grupo_creme, grupo_placebo, alternative='less', equal_var=variancias_iguais)
t_stat_bilateral, p_value_bilateral = stats.ttest_ind(grupo_creme, grupo_placebo, equal_var=variancias_iguais)

print("=== Resultados do Teste t ===")
print(f"Estatística t calculada: {t_stat:.4f}")
print(f"p-value (Unilateral - Creme < Placebo): {p_value_unilateral:.4f}")
print(f"p-value (Bilateral - Creme != Placebo): {p_value_bilateral:.4f}")

# 5. Tomada de Decisão Estatística (Nível de significância alfa = 0.05)
alpha = 0.05
print("\n=== Conclusão Estatística ===")
if p_value_unilateral < alpha:
    print(f"Como o p-value ({p_value_unilateral:.4f}) é menor que alfa ({alpha}), REJEITAMOS a hipótese nula (H0).")
    print("Há evidências estatísticas significativas de que o novo creme antirrugas é realmente eficaz na redução das rugas.")
else:
    print(f"Como o p-value ({p_value_unilateral:.4f}) é maior ou igual a alfa ({alpha}), NÃO REJEITAMOS a hipótese nula (H0).")
    print("Não há evidências estatísticas suficientes para afirmar que o novo creme é eficaz.")

# 6. Geração de Gráfico Estatístico de Visualização de Resultados
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 6))

# Boxplot para demonstrar a dispersão e mediana de cada grupo
ax = sns.boxplot(
    data=df_experimento, 
    x='Grupo', 
    y='Profundidade_Rugas_mm', 
    hue='Grupo',
    palette={"Creme Antirrugas": "#2ca02c", "Placebo": "#d62728"},
    width=0.4,
    showmeans=True,
    meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black", "markersize":"8"},
    legend=False
)

# Adicionar pontos individuais de cada voluntária com jitter (stripplot)
sns.stripplot(
    data=df_experimento, 
    x='Grupo', 
    y='Profundidade_Rugas_mm', 
    color='black', 
    alpha=0.4, 
    size=5, 
    jitter=0.1
)

# Customizações estéticas e rótulos
plt.title('Eficácia do Creme Antirrugas após 30 Dias de Uso', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Grupo de Voluntárias', fontsize=12, labelpad=10)
plt.ylabel('Profundidade das Rugas (mm)', fontsize=12, labelpad=10)

# Inserção de caixas de texto com as médias exatas no topo dos grupos
med_creme = np.mean(grupo_creme)
med_placebo = np.mean(grupo_placebo)
plt.text(0, med_creme - 0.05, f'Média: {med_creme:.2f} mm', ha='center', va='top', color='white', fontweight='bold', bbox=dict(boxstyle="round,pad=0.3", fc="#2ca02c", ec="black", lw=1))
plt.text(1, med_placebo - 0.05, f'Média: {med_placebo:.2f} mm', ha='center', va='top', color='white', fontweight='bold', bbox=dict(boxstyle="round,pad=0.3", fc="#d62728", ec="black", lw=1))

plt.tight_layout()

# Garantir que o diretório exista antes de salvar o arquivo
os.makedirs('/workspace/scratch/', exist_ok=True)
plt.savefig('/workspace/scratch/comparacao_rugas.png', dpi=150)
plt.close()

=== Estatísticas Descritivas ===
                  count      mean       std       min       25%       50%  \
Grupo                                                                       
Creme Antirrugas   30.0  1.802963  0.225002  1.371680  1.702237  1.791464   
Placebo            30.0  2.113651  0.279331  1.562099  1.937266  2.130628   

                       75%       max  
Grupo                                 
Creme Antirrugas  1.940084  2.244803  
Placebo           2.313399  2.705683  


=== Verificação de Pressupostos ===
Normalidade - Grupo Creme (p-value): 0.6868
Normalidade - Grupo Placebo (p-value): 0.9130
Igualdade de Variâncias (p-value): 0.1557


=== Resultados do Teste t ===
Estatística t calculada: -4.7444
p-value (Unilateral - Creme < Placebo): 0.0000
p-value (Bilateral - Creme != Placebo): 0.0000

=== Conclusão Estatística ===
Como o p-value (0.0000) é menor que alfa (0.05), REJEITAMOS a hipótese nula (H0).
Há evidências estatísticas significativas de que o novo crem